In [ ]:
import sys, os
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = "/content/drive/MyDrive/Duke/CS 590 RL/Project/balatro-rl-agent"
os.chdir(REPO_DIR)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

cwd: /content/balatro-rl-agent
balatro_gym importable: True
cs590_env importable:   True
save_files present:     True


In [2]:
import numpy as np
import torch
import torch.nn as nn
from functools import partial
from torch.distributions import Categorical
from dataclasses import dataclass

from gymnasium.vector import AsyncVectorEnv

from balatro_gym.constants import MAX_HAND_SIZE
from cs590_env.combat_env import (
    make_pooled_combat_env,
    VecRolloutBuffer,
    compute_gae_vectorized,
    dict_to_tensors,
    get_card_mask,
    mask_logits,
)

%run cs590_src/CombatAgent.ipynb


@dataclass
class PPOConfig:
    # ── Parallelism ──────────────────────────────────────────────
    num_envs: int         = 64
    rollout_steps: int    = 16
    # ── PPO ──────────────────────────────────────────────────────
    lr: float             = 2.5e-4
    gamma: float          = 0.99
    gae_lambda: float     = 0.95
    clip_eps: float       = 0.2
    ppo_epochs: int       = 4
    num_minibatches: int  = 4
    max_iterations: int   = 1000
    c_value: float        = 0.5
    c_entropy: float      = 0.01
    max_grad_norm: float  = 0.5
    # ── Architecture ─────────────────────────────────────────────
    d_model: int          = 256
    nhead: int            = 8
    dim_ff: int           = 1024
    dropout: float        = 0.1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cfg = PPOConfig()

In [3]:
# dict_to_tensors, get_card_mask, mask_logits
# → imported from cs590_env.combat_env (Cell 1)

In [4]:
# VecRolloutBuffer
# → imported from cs590_env.combat_env (Cell 1)

In [5]:
# compute_gae_vectorized
# → imported from cs590_env.combat_env (Cell 1)

In [6]:
def compute_log_prob_and_entropy(
    agent: CombatPPOAgent,
    obs_batch: dict,
    card_sels: torch.Tensor,
    executions: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """Forward the agent and compute log-probs / entropy for stored actions.

    Works on any batch size B (the flattened T*N during PPO, or N during rollout).

    Returns:
        log_probs: (B,)
        entropy:   (B,)
        values:    (B,)
    """
    sel_logits, exec_logits, values = agent(obs_batch)
    card_mask = get_card_mask(obs_batch)
    sel_logits = mask_logits(sel_logits, card_mask)

    sel_dist  = Categorical(logits=sel_logits)
    exec_dist = Categorical(logits=exec_logits)

    log_probs = (sel_dist.log_prob(card_sels).sum(dim=-1)
                 + exec_dist.log_prob(executions))
    entropy   = (sel_dist.entropy().sum(dim=-1)
                 + exec_dist.entropy())
    return log_probs, entropy, values.squeeze(-1)


def ppo_update(
    agent: CombatPPOAgent,
    optimizer: torch.optim.Optimizer,
    obs_batch: dict,
    card_sels: torch.Tensor,
    executions: torch.Tensor,
    old_log_probs: torch.Tensor,
    advantages: torch.Tensor,
    returns: torch.Tensor,
    cfg: PPOConfig,
) -> dict:
    """Run PPO clipped-surrogate epochs over flattened (T*N) rollout data.

    Returns:
        dict of mean losses for logging.
    """
    B = len(advantages)
    mb_size = B // cfg.num_minibatches
    total_pg, total_vf, total_ent = 0.0, 0.0, 0.0
    num_updates = 0

    for _ in range(cfg.ppo_epochs):
        indices = torch.randperm(B, device=advantages.device)

        for start in range(0, B, mb_size):
            end = start + mb_size
            if end > B:
                break
            mb = indices[start:end]

            mb_obs          = {k: v[mb] for k, v in obs_batch.items()}
            mb_card_sels    = card_sels[mb]
            mb_executions   = executions[mb]
            mb_old_lp       = old_log_probs[mb]
            mb_adv          = advantages[mb]
            mb_ret          = returns[mb]

            curr_lp, entropy, curr_values = compute_log_prob_and_entropy(
                agent, mb_obs, mb_card_sels, mb_executions)

            ratio = torch.exp(curr_lp - mb_old_lp)
            surr1 = ratio * mb_adv
            surr2 = torch.clamp(ratio, 1 - cfg.clip_eps, 1 + cfg.clip_eps) * mb_adv
            pg_loss    = -torch.min(surr1, surr2).mean()
            value_loss = nn.functional.mse_loss(curr_values, mb_ret)
            ent_bonus  = entropy.mean()

            loss = pg_loss + cfg.c_value * value_loss - cfg.c_entropy * ent_bonus

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(agent.parameters(), cfg.max_grad_norm)
            optimizer.step()

            total_pg  += pg_loss.item()
            total_vf  += value_loss.item()
            total_ent += ent_bonus.item()
            num_updates += 1

    return {
        'pg_loss':    total_pg  / max(num_updates, 1),
        'value_loss': total_vf  / max(num_updates, 1),
        'entropy':    total_ent / max(num_updates, 1),
    }

In [7]:
# ── Snapshot pool from .jkr save files ────────────────────────────

from pathlib import Path
from balatro_gym.save_injection import inject_save_into_balatro_env

def load_snapshot_pool(save_dir: str = "save_files", seed: int = 42) -> list[dict]:
    """Load .jkr save files and convert them into env snapshots."""
    pool = []
    for jkr_path in sorted(Path(save_dir).glob("*.jkr")):
        env, report = inject_save_into_balatro_env(jkr_path, seed=seed)
        pool.append(env.save_state())
    if not pool:
        raise FileNotFoundError(f"No .jkr files found in {save_dir}/")
    return pool

snapshot_pool = load_snapshot_pool()

# ── Initialise agent, vec-env, optimizer, buffer ─────────────────

agent = CombatPPOAgent(
    d_model=cfg.d_model, nhead=cfg.nhead,
    dim_ff=cfg.dim_ff, dropout=cfg.dropout,
).to(device)

optimizer = torch.optim.Adam(agent.parameters(), lr=cfg.lr, eps=1e-5)
buffer    = VecRolloutBuffer(cfg.rollout_steps, cfg.num_envs, device)

BASE_SEED = 12345
vec_env = AsyncVectorEnv([
    partial(make_pooled_combat_env, snapshot_pool, pool_seed=BASE_SEED + i)
    for i in range(cfg.num_envs)
])

# ── Training loop ────────────────────────────────────────────────

obs_np, _ = vec_env.reset()          # dict of (N, ...) arrays
ep_returns: list[float] = []
ep_lengths: list[int]   = []
running_rewards = np.zeros(cfg.num_envs)
running_lens    = np.zeros(cfg.num_envs, dtype=int)

for iteration in range(1, cfg.max_iterations + 1):

    # ── Phase 1: Parallel rollout collection ─────────────────────
    agent.eval()
    for t in range(cfg.rollout_steps):
        obs_t = dict_to_tensors(obs_np, device)         # {key: (N, ...)} on GPU
        card_mask = get_card_mask(obs_t)

        with torch.no_grad():
            sel_logits, exec_logits, value = agent(obs_t)

        sel_logits = mask_logits(sel_logits, card_mask)

        sel_dist  = Categorical(logits=sel_logits)       # (N, MAX_HAND_SIZE, 2)
        exec_dist = Categorical(logits=exec_logits)      # (N, 2)

        card_sels  = sel_dist.sample()                   # (N, MAX_HAND_SIZE)
        executions = exec_dist.sample()                  # (N,)

        log_probs = (sel_dist.log_prob(card_sels).sum(dim=-1)
                     + exec_dist.log_prob(executions))   # (N,)

        # Encode as flat (N, MAX_HAND_SIZE+1) action for the vec env
        actions_np = np.concatenate([
            card_sels.cpu().numpy(),
            executions.cpu().numpy()[:, None],
        ], axis=-1)

        next_obs_np, rewards_np, terms, truncs, infos = vec_env.step(actions_np)
        dones_np = np.logical_or(terms, truncs)

        buffer.store_step(
            t, obs_t, card_sels, executions, log_probs,
            value.squeeze(-1), rewards_np, dones_np,
        )

        running_rewards += rewards_np
        running_lens    += 1
        for i in np.where(dones_np)[0]:
            ep_returns.append(running_rewards[i])
            ep_lengths.append(running_lens[i])
            running_rewards[i] = 0.0
            running_lens[i]    = 0

        obs_np = next_obs_np

    # ── Phase 2: Per-env GAE + PPO update ────────────────────────
    with torch.no_grad():
        _, _, next_vals = agent(dict_to_tensors(obs_np, device))
        next_vals = next_vals.squeeze(-1)                # (N,)

    advantages, returns = compute_gae_vectorized(
        buffer.rewards, buffer.values, next_vals, buffer.dones,
        cfg.gamma, cfg.gae_lambda,
    )

    flat_obs, flat_card_sels, flat_execs, flat_lp = buffer.flatten()
    flat_adv = advantages.reshape(-1)
    flat_ret = returns.reshape(-1)

    agent.train()
    losses = ppo_update(
        agent, optimizer,
        flat_obs, flat_card_sels, flat_execs, flat_lp,
        flat_adv, flat_ret, cfg,
    )

    # ── Logging ──────────────────────────────────────────────────
    if iteration % 10 == 0 or iteration == 1:
        recent_r   = np.mean(ep_returns[-50:]) if ep_returns else 0.0
        recent_len = np.mean(ep_lengths[-50:]) if ep_lengths else 0.0
        print(f"[iter {iteration:4d}]  "
              f"reward={recent_r:+.2f}  len={recent_len:.1f}  "
              f"pg={losses['pg_loss']:.4f}  "
              f"vf={losses['value_loss']:.4f}  "
              f"ent={losses['entropy']:.4f}  "
              f"episodes={len(ep_returns)}")

vec_env.close()

=== OBS RANGE DIAGNOSTIC (all envs) ===
  hand_card_ids                   min=   -1  max=   50  shape=(64, 10)              (max card_id for rank/suit) *** OOB: min=-1 < 0 ***
  hand_card_enhancements          min=    0  max=    7  shape=(64, 10)              (CardEmbedding._NUM_ENHANCEMENTS)
  hand_card_editions              min=    0  max=    0  shape=(64, 10)              (CardEmbedding._NUM_EDITIONS)
  hand_card_seals                 min=    0  max=    2  shape=(64, 10)              (CardEmbedding._NUM_SEALS)
  deck_card_ids                   min=   -1  max=   51  shape=(64, 100)             (max card_id for rank/suit) *** OOB: min=-1 < 0 ***
  deck_card_enhancements          min=   -1  max=    8  shape=(64, 100)             (CardEmbedding._NUM_ENHANCEMENTS) *** OOB: min=-1 < 0 ***
  deck_card_editions              min=   -1  max=    0  shape=(64, 100)             (CardEmbedding._NUM_EDITIONS) *** OOB: min=-1 < 0 ***
  deck_card_seals                 min=   -1  max=    4  shape=(64

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
